# Confidence Intervals
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Construct** a confidence interval for a population mean given a sample
- **Interpret** what "95% confidence" actually means, and what it does not mean
- **Explain** how sample size and confidence level both affect the width of the interval

> **Tip:** Start by moving the **confidence level slider** from 90% to 99% and watching the interval widen, then move the **sample size slider** to see how more data narrows it.

---
## How we got here

In *14: Sampling Distributions* we learned that sample means vary around the true population mean with a spread equal to the standard error. In *15: Z-Scores & Standardization* we learned to translate distances from the mean into probabilities using z-scores. A confidence interval combines both ideas: it uses the z-score for the desired confidence level and the standard error to build a range that captures the population parameter with a stated probability.

---
## Why this matters for data science

Every model evaluation involves uncertainty. When you report "our model achieves 84% accuracy on the test set," a confidence interval turns that into "we are 95% confident the true accuracy is between 81% and 87%." This is the difference between a number and a claim you can stand behind. A/B testing, clinical trials, and survey research all report confidence intervals as standard practice.

---
## Try it yourself

In [2]:
from plotly.subplots import make_subplots
from ipywidgets import HTML
from tkh_utils import make_int_slider, make_dropdown

_TRUE_MU    = 50.0
_TRUE_SIGMA = 10.0
_X_RANGE    = [_TRUE_MU - 4.5 * _TRUE_SIGMA, _TRUE_MU + 4.5 * _TRUE_SIGMA]
_Z_VALS     = {"80%": 1.282, "90%": 1.645, "95%": 1.960, "99%": 2.576}
_MAX_SHOW   = 60   # CIs visible in history panel at once

_state = {"cis": [], "last_sample": None}

# ── Persistent FigureWidget — built once, updated in-place ────────────────────
_sub = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Current Sample", "CI History  (newest at top)"],
    column_widths=[0.35, 0.65],
    horizontal_spacing=0.10,
)

# Panel 1: current sample view
# All traces start with x=[], y=[] — multi-element initial arrays trigger a
# numpy truth-value error in ipywidgets' state sync (if not input_val:)
_sub.add_trace(go.Scatter(  # T0: sample dots
    x=[], y=[], mode="markers", showlegend=False,
    marker=dict(color=PALETTE["secondary"], size=7, opacity=0.65,
                line=dict(color="white", width=1)),
), row=1, col=1)
_sub.add_trace(go.Scatter(  # T1: CI line
    x=[], y=[], mode="lines", showlegend=False,
    line=dict(color=PALETTE["accent"], width=6),
), row=1, col=1)
_sub.add_trace(go.Scatter(  # T2: sample mean marker
    x=[], y=[], mode="markers", showlegend=False,
    marker=dict(color=PALETTE["accent"], symbol="diamond", size=13,
                line=dict(color="white", width=1.5)),
), row=1, col=1)

# Panel 2: running CI history
_sub.add_trace(go.Scatter(  # T3: captured CI lines
    x=[], y=[], mode="lines", showlegend=True,
    line=dict(color="#27ae60", width=2.5),
    name="✓ Captured μ",
), row=1, col=2)
_sub.add_trace(go.Scatter(  # T4: missed CI lines
    x=[], y=[], mode="lines", showlegend=True,
    line=dict(color="#e74c3c", width=2.5),
    name="✗ Missed μ",
), row=1, col=2)
_sub.add_trace(go.Scatter(  # T5: mean dots for captured
    x=[], y=[], mode="markers", showlegend=False,
    marker=dict(color="#27ae60", size=5),
), row=1, col=2)
_sub.add_trace(go.Scatter(  # T6: mean dots for missed
    x=[], y=[], mode="markers", showlegend=False,
    marker=dict(color="#e74c3c", size=5),
), row=1, col=2)
_sub.add_trace(go.Scatter(  # T7: true μ vertical reference — float values avoid int-array sync issue
    x=[_TRUE_MU, _TRUE_MU], y=[-1.0, 6.0],
    mode="lines", showlegend=True,
    line=dict(color="#333", width=1.5, dash="dash"),
    name=f"μ = {_TRUE_MU:.0f}",
), row=1, col=2)

_bl = base_layout(
    title="Confidence Intervals — what does '95% confidence' actually mean?",
    xaxis_title="", yaxis_title="",
)
_bl.update(height=500, margin=dict(t=90, b=50, l=50, r=30), showlegend=True)
_sub.update_layout(_bl)
_sub.update_xaxes(title_text="Value", range=_X_RANGE, row=1, col=1)
_sub.update_yaxes(visible=False, range=[-0.55, 0.55], row=1, col=1)
_sub.update_xaxes(title_text="Value", range=_X_RANGE, row=1, col=2)
_sub.update_yaxes(title_text="Sample #", showticklabels=False, row=1, col=2)

fig_widget    = go.FigureWidget(_sub)
_n_fixed_anns = len(fig_widget.layout.annotations)   # 2 subplot titles

# ── Controls ──────────────────────────────────────────────────────────────────
n_sl         = make_int_slider(5, 200, 5, 30,  "Sample size (n):")
conf_dd      = make_dropdown(["80%", "90%", "95%", "99%"], "95%", "Confidence level:")
draw_one_btn = Button(
    description="Draw 1 sample", button_style="primary", icon="arrow-right",
    layout=widgets.Layout(width="160px", height="34px"),
)
draw_ten_btn = Button(
    description="Draw 10", icon="forward",
    layout=widgets.Layout(width="110px", height="34px"),
)
reset_btn = Button(
    description="Reset", button_style="warning", icon="undo",
    layout=widgets.Layout(width="100px", height="34px"),
)
summary_html = HTML(
    value=(
        f'<p style="font-family:{FONT["family"]};color:#888;margin:6px 0;">'
        f'Population: Normal(μ=50, σ=10).  '
        f'Draw samples to track how often a 95% CI captures μ = 50.</p>'
    )
)

# ── Render ─────────────────────────────────────────────────────────────────────
def render():
    cis       = _state["cis"]
    last_samp = _state["last_sample"]
    n         = n_sl.value
    conf      = conf_dd.value
    z_star    = _Z_VALS[conf]
    num_drawn = len(cis)
    cis_shown = cis[-_MAX_SHOW:]
    n_shown   = len(cis_shown)

    with fig_widget.batch_update():
        # T0: sample dots — convert numpy arrays to lists to avoid state-sync errors
        if last_samp is not None:
            jitter = np.random.default_rng(42).uniform(-0.38, 0.38, len(last_samp))
            fig_widget.data[0].x = last_samp.tolist()
            fig_widget.data[0].y = jitter.tolist()
        else:
            fig_widget.data[0].x = []
            fig_widget.data[0].y = []

        # T1: CI line — always clear both x and y in the empty branch
        if cis:
            xm, lo, hi, cap = cis[-1]
            fig_widget.data[1].x          = [lo, hi]
            fig_widget.data[1].y          = [0.0, 0.0]
            fig_widget.data[1].line.color = "#27ae60" if cap else "#e74c3c"
        else:
            fig_widget.data[1].x = []
            fig_widget.data[1].y = []

        # T2: mean marker — always clear both x and y in the empty branch
        if cis:
            xm, lo, hi, cap = cis[-1]
            fig_widget.data[2].x            = [xm]
            fig_widget.data[2].y            = [0.0]
            fig_widget.data[2].marker.color = "#27ae60" if cap else "#e74c3c"
        else:
            fig_widget.data[2].x = []
            fig_widget.data[2].y = []

        # T3–T6: CI history (panel 2) — newest at highest y (top)
        xs_cap, ys_cap   = [], []
        xs_mis, ys_mis   = [], []
        xs_mcap, ys_mcap = [], []
        xs_mmis, ys_mmis = [], []

        for i, (xm, lo, hi, cap) in enumerate(cis_shown):
            if cap:
                xs_cap.extend([lo, hi, None]);  ys_cap.extend([float(i), float(i), None])
                xs_mcap.append(xm);             ys_mcap.append(float(i))
            else:
                xs_mis.extend([lo, hi, None]);  ys_mis.extend([float(i), float(i), None])
                xs_mmis.append(xm);             ys_mmis.append(float(i))

        fig_widget.data[3].x = xs_cap;  fig_widget.data[3].y = ys_cap
        fig_widget.data[4].x = xs_mis;  fig_widget.data[4].y = ys_mis
        fig_widget.data[5].x = xs_mcap; fig_widget.data[5].y = ys_mcap
        fig_widget.data[6].x = xs_mmis; fig_widget.data[6].y = ys_mmis

        # T7: true μ line — stretch to current history height
        y_hi = float(max(n_shown + 1, 6))
        fig_widget.data[7].y           = [-1.0, y_hi]
        fig_widget.layout.yaxis2.range = [-1.0, y_hi]

        # Annotations: preserve subplot titles + add panel-1 status label
        anns = list(fig_widget.layout.annotations)[:_n_fixed_anns]

        if not cis:
            anns.append(dict(
                text='Click "Draw 1 sample" to draw your first sample',
                xref="x domain", yref="y domain",
                x=0.5, y=0.5, xanchor="center", yanchor="middle",
                showarrow=False,
                font=dict(size=12, family=FONT["family"], color="#aaa"),
            ))
        else:
            xm, lo, hi, cap = cis[-1]
            color = "#27ae60" if cap else "#e74c3c"
            word  = "✓ captured" if cap else "✗ missed"
            anns.append(dict(
                text=f"<b>x̄ = {xm:.2f}</b>  {word}<br>CI: [{lo:.2f}, {hi:.2f}]",
                xref="x domain", yref="y domain",
                x=0.5, y=0.97, xanchor="center", yanchor="top",
                showarrow=False,
                font=dict(size=11, family=FONT["family"], color=color),
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor=color, borderwidth=1, borderpad=4,
            ))

        fig_widget.layout.annotations = anns

    # Summary bar (outside batch_update — separate widget)
    se    = _TRUE_SIGMA / np.sqrt(n)
    width = 2 * z_star * se
    if num_drawn > 0:
        n_cap  = sum(1 for _, _, _, cap in cis if cap)
        pct    = 100 * n_cap / num_drawn
        target = float(conf[:-1])
        nc     = "#27ae60" if abs(pct - target) < 8 else "#e67e22"
        summary_html.value = (
            f'<div style="font-family:{FONT["family"]};background:#f8f9fb;'
            f'border-left:4px solid {PALETTE["primary"]};border-radius:6px;'
            f'padding:10px 16px;margin-top:4px;font-size:13px;line-height:1.8;">'
            f'<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:16px;">'
            f'<div><b>Setup</b><br>n = {n},  {conf} CI<br>'
            f'z* = {z_star:.3f},  σ = {_TRUE_SIGMA:.1f}</div>'
            f'<div><b>Interval width</b><br>2 × z* × σ/√n<br>'
            f'= 2 × {z_star:.3f} × {_TRUE_SIGMA:.1f}/√{n} = <b>{width:.3f}</b></div>'
            f'<div><b>Capture rate</b><br>{n_cap} / {num_drawn} intervals<br>'
            f'<span style="color:{nc};font-weight:600;">'
            f'{pct:.1f}% captured μ = {_TRUE_MU:.0f}</span></div>'
            f'</div></div>'
        )
    else:
        summary_html.value = (
            f'<p style="font-family:{FONT["family"]};color:#888;margin:6px 0;">'
            f'Population: Normal(μ={_TRUE_MU:.0f}, σ={_TRUE_SIGMA:.0f}).  '
            f'Each {conf} CI has width {width:.3f}.  '
            f'Draw samples to track how often μ is captured.</p>'
        )

# ── Event handlers ─────────────────────────────────────────────────────────────
def _draw_one(n, conf):
    z_star = _Z_VALS[conf]
    sample = np.random.normal(_TRUE_MU, _TRUE_SIGMA, n)
    xm     = float(sample.mean())
    se     = _TRUE_SIGMA / np.sqrt(n)
    lo, hi = xm - z_star * se, xm + z_star * se
    _state["last_sample"] = sample
    _state["cis"].append((xm, lo, hi, lo <= _TRUE_MU <= hi))

def _on_draw_one(b):
    _draw_one(n_sl.value, conf_dd.value)
    render()

def _on_draw_ten(b):
    for _ in range(10):
        _draw_one(n_sl.value, conf_dd.value)
    render()

def _on_reset(b):
    _state["cis"]         = []
    _state["last_sample"] = None
    render()

def _on_param_change(change):
    _state["cis"]         = []
    _state["last_sample"] = None
    render()

draw_one_btn.on_click(_on_draw_one)
draw_ten_btn.on_click(_on_draw_ten)
reset_btn.on_click(_on_reset)
n_sl.observe(_on_param_change,    names="value")
conf_dd.observe(_on_param_change, names="value")

# ── Display — fig_widget in VBox directly, no Output() wrapper ────────────────
btn_row  = HBox([draw_one_btn, draw_ten_btn, reset_btn],
                layout=widgets.Layout(gap="8px", margin="4px 0"))
controls = VBox([n_sl, conf_dd, btn_row])
display(VBox([controls, fig_widget, summary_html]))
render()


---
## What's happening?

A confidence interval is a range of values, computed from sample data, that contains the true population parameter with a stated probability.

| Symbol | Meaning |
|--------|---------|
| x̄ | Sample mean |
| σ | Population standard deviation (or s for sample) |
| n | Sample size |
| z* | Critical z-value for the desired confidence level |
| CI | Confidence interval |

$$CI = \bar{x} \pm z^* \cdot \frac{\sigma}{\sqrt{n}}$$

Common critical values: z* = 1.645 (90%), z* = 1.96 (95%), z* = 2.576 (99%)

### What "95% confident" really means

It does **not** mean: "there is a 95% chance the true mean is in this interval." The true mean is a fixed (if unknown) number, it either is or is not in your interval. What it means: if you repeated this sampling process many times and built a CI each time, **95% of those intervals would contain the true mean**.

The widget's running count demonstrates this exactly, watch for the capture rate to hover near your chosen confidence level.

---
## Real-world example: Margin of error in political polling

Every election poll includes a margin of error, that is a confidence interval. A poll reporting "52% ± 3 points at 95% confidence" is saying the true support lies between 49% and 55%, with 95% confidence.

The chart simulates 50 polls of n = 600 voters from a population where true support is 52%. Each horizontal line is one poll's 95% CI. Notice:

- **Notice:** Most intervals contain the true value (52%), but a few miss, roughly 1 in 20, consistent with 95% confidence
- **Notice:** The intervals vary in center (the poll result) and some are entirely above or below the true value, those polls got unlucky
- **Notice:** All intervals have the same width, because n and the confidence level are fixed for every poll

> **Discussion question:** A campaign sees a poll with CI [49.5%, 54.5%] and concludes they are winning. A rival poll shows [47%, 53%]. Are both polls consistent with the same underlying truth? What would you tell the campaign?

In [3]:
np.random.seed(101)

# ── Confidence interval simulation for election polling ───────────────────────
true_p  = 0.52
n_poll  = 600
n_polls = 50
z_star  = 1.96   # 95% CI

polls = np.random.binomial(n_poll, true_p, n_polls) / n_poll
se    = np.sqrt(polls * (1 - polls) / n_poll)
lower = polls - z_star * se
upper = polls + z_star * se
captured = (lower <= true_p) & (upper >= true_p)
capture_rate = captured.mean()

colors = [PALETTE["accent"] if c else PALETTE["secondary"] for c in captured]

fig = go.Figure()
for i, (lo, hi, col) in enumerate(zip(lower, upper, colors)):
    fig.add_shape(type="line", x0=lo*100, x1=hi*100, y0=i, y1=i,
                  line=dict(color=col, width=2))
    fig.add_shape(type="line", x0=lo*100, x1=lo*100, y0=i-0.3, y1=i+0.3,
                  line=dict(color=col, width=1.5))
    fig.add_shape(type="line", x0=hi*100, x1=hi*100, y0=i-0.3, y1=i+0.3,
                  line=dict(color=col, width=1.5))

fig.add_vline(x=true_p*100, line_color=PALETTE["primary"], line_width=2.5,
              annotation_text=f"True p = {true_p*100:.0f}%")
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color=PALETTE["accent"]), name=f"Captured true p ({captured.sum()}/{n_polls})"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color=PALETTE["secondary"]), name="Missed true p"))

layout = base_layout(
    title=f"95% Confidence Intervals — {n_polls} Polls  (capture rate: {capture_rate:.0%})",
    xaxis_title="Poll Result (%)",
    yaxis_title="Poll Number",
)
layout.update(yaxis=dict(range=[-1, n_polls + 1]))
fig.update_layout(layout)
fig.show()

### How sample size and confidence level affect interval width

| Confidence level | z* | n = 100 (width) | n = 400 (width) | n = 1000 (width) |
|-----------------|-----|----------------|----------------|-----------------|
| 80% | 1.282 | ±12.8% | ±6.4% | ±4.1% |
| 90% | 1.645 | ±16.5% | ±8.2% | ±5.2% |
| 95% | 1.960 | ±19.6% | ±9.8% | ±6.2% |
| 99% | 2.576 | ±25.8% | ±12.9% | ±8.1% |

*(Width shown for proportion p = 0.5, which maximizes SE)*

---
## Key takeaway

> **A confidence interval is not a guarantee that the true value is inside; it is a procedure that captures the true value at the stated rate across repeated samples — wider intervals offer more confidence but less precision.**

---
*Next up: Hypothesis Testing — using the same sampling distribution logic to make binary decisions: is an effect real or just chance?*